# CFM — tous les modèles en un seul notebook

Un seul « Run All » raconte le projet, du plus simple au meilleur. Protocole identique partout :
- `fit` entraîne ;
- `valid` (grappes de régimes tenues à l'écart) choisit ;
- `stress` (carnets peu profonds, comme le test) diagnostique ;
- la précision **équilibrée** applique la correction de Sinkhorn.

| Partie | Question | Durée GPU estimée |
|---|---|---|
| 1 · Benchmark | Que vaut-on sans rien d'intelligent ? (hasard, majorité, Naive Bayes, logistique, kNN, forêt aléatoire) | 5–10 min |
| 2 · Arbres | Les features du papier aident-elles **le même** XGBoost ? LightGBM, CatBoost, importances, utilité/dérive, AUC adversariale | 10–20 min |
| 3 · Séquences | MLP, CNN, GRU, petit Transformer sur l'**ancienne** représentation | ~15 min |
| 4 · Le 0,507 | Reconstruction simplifiée de l'hybride V2 | ~15 min |
| 5 · Améliorations | Un petit CNN, puis chaque amélioration du papier ajoutée une à une | 15–20 min |
| Synthèse | Tous les modèles sur un même graphique | < 1 min |

Total : environ 1 h à 1 h 20 sur GPU, non mesuré. Le cache est construit une fois (≈ 20 min), ou réutilisé si `lab_v3` existe.
`DEMO = True` : données synthétiques sur CPU, quelques minutes, pour vérifier la mécanique (scores sans valeur).

In [ ]:
import os, sys, json, time
from pathlib import Path
if sys.platform == 'darwin':
    os.environ.setdefault('OMP_NUM_THREADS', '1')
CODE_DATASET = None   # ex. 'ton-username/cfm-signature-lab' ; None : code local ou Dataset attaché

def find_repo():
    if CODE_DATASET:
        import kagglehub
        path = Path(kagglehub.dataset_download(CODE_DATASET, force_download=True))
        return next(p.parent.parent for p in path.glob('**/cfm/__init__.py'))
    for p in [Path('.'), Path('..'), Path('../..'), Path('/kaggle/working/cfm-signature-lab')]:
        if (p / 'cfm' / '__init__.py').exists():
            return p.resolve()
    for p in Path('/kaggle/input').glob('**/cfm/__init__.py'):
        return p.parent.parent
    raise FileNotFoundError('Code introuvable : renseigner CODE_DATASET ou attacher le Dataset du code')

REPO = find_repo(); sys.path.insert(0, str(REPO))
DEMO = False          # True : données synthétiques sur CPU (vérifie que tout tourne ; scores sans valeur)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from cfm.demo import core, models, viz, train as T
import sys
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from cfm import tabular, screen, plots as cplots
from cfm.registry import BLOCKS
from cfm.blend import sinkhorn_balance
from cfm import transductive as TR
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
viz.setup('figures/models')
lab = core.open_lab(DEMO)
LABELS = [str(c) for c in lab.classes]
RESULTS = []

## Partie 1 · Benchmark — Benchmark : que vaut-on sans rien d'intelligent ?

**Question.** Avant tout réseau, quels scores obtient-on avec des modèles classiques sur les **statistiques de fenêtre
de la première version** (moyennes et écarts-types de canaux relatifs + fréquences des catégories) ?

**Protocole commun à tout `v_demo`.**
- `fit` entraîne.
- `valid` (grappes de régimes tenues à l'écart) choisit.
- `stress` (carnets peu profonds, comme le test) diagnostique.
- Le hasard vaut 1/24 ≈ 0,042.

Durée : environ 5 à 10 min (CPU suffit).

In [ ]:
X, names = lab.table(core.OLD_STATS)
fit, valid, stress = lab.split['fit'], lab.split['valid'], lab.split['stress']
med = np.nanmedian(X[fit], 0); X = np.where(np.isfinite(X), X, med)
sc = StandardScaler().fit(X[fit]); Xs = sc.transform(X)
print(f'{X.shape[1]} statistiques par fenêtre :', names[:6], '…')

In [ ]:
rows, probs = [], {}
def run(nom, make, Xm=Xs, sub=None):
    t0 = time.time(); tr = fit if sub is None else np.random.default_rng(0).choice(fit, min(sub, len(fit)), replace=False)
    m = make().fit(Xm[tr], lab.y[tr])
    out = {'modèle': nom}
    for part, idx in [('valid', valid), ('stress', stress)]:
        p = m.predict_proba(Xm[idx]); probs[(nom, part)] = p
        out[part] = core.accuracy(lab.y[idx], p); out[part + '_équilibré'] = core.accuracy(lab.y[idx], p, True)
    out['minutes'] = round((time.time() - t0) / 60, 2); rows.append(out); print(out, flush=True)

class Majority:
    def fit(self, X, y): self.p = np.bincount(y, minlength=len(LABELS)) / len(y); return self
    def predict_proba(self, X): return np.tile(self.p, (len(X), 1))

run('Classe majoritaire', Majority)
run('Naive Bayes gaussien', GaussianNB)
run('Régression logistique', lambda: LogisticRegression(max_iter=300, C=1.0))
run('k plus proches voisins (k=25)', lambda: KNeighborsClassifier(25, n_jobs=-1), sub=40000)
run('Forêt aléatoire (200 arbres)', lambda: RandomForestClassifier(200, max_depth=20, min_samples_leaf=3, n_jobs=-1, random_state=0), Xm=X)
res = pd.DataFrame(rows); res

In [ ]:
RESULTS.append(res.assign(partie='Partie 1 · Benchmark'))

In [ ]:
viz.compare(res, ('valid', 'stress'), 'V1 — modèles classiques sur les statistiques de la première version', 'v1_compare')
best = res.sort_values('valid', ascending=False).iloc[0]['modèle']
viz.confusion(lab.y[valid], probs[(best, 'valid')], LABELS, f'Confusion — {best} (valid)', 'v1_confusion')
viz.recall_by_class(lab.y[valid], {best: probs[(best, 'valid')]}, LABELS, f'Rappel par titre — {best}', 'v1_recall')

**À retenir.**
- Ces scores sont le **plancher**.
- Des statistiques relatives, même avec un bon modèle d'arbres, plafonnent bas.
- L'écart entre `valid` et `stress` montre déjà le problème du changement de régime.

La suite (V2) teste si de **meilleures features** relèvent ce plafond, avant d'aller vers les séquences (V3).

## Partie 2 · Arbres — Modèles d'arbres : l'effet des features

**Question.** Avec le même modèle (XGBoost), combien rapportent les features du papier (ticks, lots, position,
parcours d'ordres…) par rapport aux statistiques relatives de la première version ? Puis LightGBM et CatBoost
sur l'ensemble complet, l'importance des features, et la carte utilité/dérive.

Durée : environ 10 à 20 min sur GPU.

In [ ]:
ALL = ['base_relative', 'cat_freq', 'cat_cross', 'levels', 'ticks', 'lots', 'position', 'orders', 'book_dyn', 'tokens_bow', 'transitions']
fit, valid, stress = lab.split['fit'], lab.split['valid'], lab.split['stress']
tr, es = tabular.inner_split(fit, lab.y, 0)          # arrêt anticipé sur une tranche interne du fit
DEV = 'cuda' if core.has_cuda() else 'cpu'
rows, probs, importances = [], {}, {}

def run(nom, blocks, model='xgb', params=None):
    X, names = lab.table(blocks); t0 = time.time()
    if model == 'catboost':
        from catboost import CatBoostClassifier
        m = CatBoostClassifier(iterations=1500, learning_rate=.1, depth=6, loss_function='MultiClass', verbose=0,
                               task_type='GPU' if DEV == 'cuda' else 'CPU', random_seed=0, early_stopping_rounds=50)
        m.fit(X[tr], lab.y[tr], eval_set=(X[es], lab.y[es])); pred = m.predict_proba
    else:
        pred, _ = tabular.train(X, lab.y, tr, es, model, params, 0, DEV if model == 'xgb' else 'cpu')
        if model == 'xgb':
            m = pred.__self__; importances[nom] = pd.Series(m.feature_importances_, index=names)
    out = {'modèle': nom, 'features': X.shape[1]}
    for part, idx in [('valid', valid), ('stress', stress)]:
        p = pred(X[idx]); probs[(nom, part)] = p
        out[part] = core.accuracy(lab.y[idx], p); out[part + '_équilibré'] = core.accuracy(lab.y[idx], p, True)
    out['minutes'] = round((time.time() - t0) / 60, 1); rows.append(out); print(out, flush=True)

In [ ]:
run('XGBoost — stats première version', core.OLD_STATS)
run('XGBoost — + ticks', core.OLD_STATS + ['ticks'])
run('XGBoost — + ticks + lots', core.OLD_STATS + ['ticks', 'lots'])
run('XGBoost — + ticks + lots + position + ordres', core.OLD_STATS + ['ticks', 'lots', 'position', 'orders'])
run('XGBoost — toutes les features', ALL)
if sys.platform != 'darwin':   # sous macOS, LightGBM + torch dans un même processus peut planter (OpenMP)
    run('LightGBM — toutes les features', ALL, 'lgbm')
run('CatBoost — toutes les features', ALL, 'catboost')
res = pd.DataFrame(rows); res

In [ ]:
RESULTS.append(res.assign(partie='Partie 2 · Arbres'))

In [ ]:
viz.compare(res, ('valid', 'stress'), 'V2 — arbres : chaque famille de features ajoutée une à une', 'v2_compare')
imp = importances['XGBoost — toutes les features'].sort_values(ascending=False).head(25)
fam = [BLOCKS[n.split(':')[0]].family for n in imp.index]
cols = {f: viz.CYCLE[i % len(viz.CYCLE)] for i, f in enumerate(sorted(set(fam)))}
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(range(len(imp)), imp.values, color=[cols[f] for f in fam]); ax.set_yticks(range(len(imp)), imp.index, fontsize=7); ax.invert_yaxis()
for f, c in cols.items(): ax.barh([], [], color=c, label=f)
ax.legend(fontsize=7); ax.set_title('25 features les plus utilisées par XGBoost (gain)'); viz.save(fig, 'v2_importance')

In [ ]:
# Utilité (η², sur le fit) contre dérive (KS train/test) : la carte qui a guidé tout le projet
uni = screen.univariate(lab.dir, lab.cfg, ALL, base=core.OLD_STATS)
cplots.screen_map(uni, lab.dir); fig = plt.gcf(); viz.save(fig, 'v2_utility_vs_drift')
uni.head(15)[['feature', 'family', 'eta2', 'ks', 'shift_sd']].round(3)

In [ ]:
# Où se trouve le décalage train/test ? AUC d'un classifieur train-vs-test, bloc par bloc
adv = screen.adversarial(lab.dir, lab.cfg, ALL, n=20000, device=DEV)
cplots.adversarial_bars(adv, lab.dir); viz.save(plt.gcf(), 'v2_adversarial'); adv.round(3)

**À retenir.**
- Ajouter les ticks et les lots fait monter **le même XGBoost** de plusieurs points : c'est la représentation qui compte.
- Les arbres plafonnent pourtant loin des réseaux : ils voient des **fréquences**, pas l'**ordre** des événements (V3).
- La carte utilité/dérive montre les colonnes de tick en haut à gauche : séparatrices **et** stables.

## Partie 3 · Séquences — Modèles séquentiels sur l'ancienne représentation

**Question.** Lire les 100 événements dans l'ordre apporte-t-il quelque chose par rapport aux seules statistiques ?
Tous les modèles ont ici la **même entrée que la première version** : canaux relatifs à des médianes locales
+ 4 catégories (venue, action, côté, trade). Aucun tick, aucun lot.

Durée : 4 modèles × 15 époques, environ 15 min sur GPU.

In [ ]:
data = lab.sequences('old')
print('tokens :', core.OLD_TOKENS, '| canaux :', data['cont_names'], '| contexte :', data['ctx'].shape[1], 'colonnes')
EPOCHS = 3 if DEMO else 15
zoo = {
    'MLP sur statistiques': lambda: models.StatsMLP(data['ctx'].shape[1]),
    'CNN (3 conv. dilatées)': lambda: models.TokenCNN(data['cards'], data['cont'].shape[-1], data['ctx'].shape[1]),
    'GRU bidirectionnel': lambda: models.TinyGRU(data['cards'], data['cont'].shape[-1], data['ctx'].shape[1]),
    'Petit Transformer': lambda: models.TinyTransformer(data['cards'], data['cont'].shape[-1], data['ctx'].shape[1]),
}
rows, hists, probs = [], {}, {}
for nom, make in zoo.items():
    print('▶', nom); t0 = time.time()
    m, h, norm, dev = T.train(make(), lab, data, epochs=EPOCHS)
    out, p = T.evaluate(m, lab, data, norm, dev, nom, time.time() - t0)
    rows.append(out); hists[nom] = h; probs[nom] = p
res = pd.DataFrame(rows); res.round(4)

In [ ]:
RESULTS.append(res.assign(partie='Partie 3 · Séquences'))

In [ ]:
viz.compare(res, ('valid', 'stress'), 'V3 — lire la séquence (ancienne représentation)', 'v3_compare')
viz.curves(hists, 'V3 — apprentissage', 'v3_curves')
viz.recall_by_class(lab.y[lab.split['valid']], {k: v['valid'] for k, v in probs.items()}, LABELS, 'Rappel par titre (valid)', 'v3_recall')

**À retenir.**
- Passer des statistiques (MLP) à la lecture de la séquence (CNN, GRU, Transformer) fait gagner beaucoup.
- L'information est dans **l'enchaînement des événements**.
- Mais tous ces modèles voient encore des grandeurs relatives à la fenêtre. V4 reconstruit le meilleur modèle de cette famille (0,507). V5 montre ce que change la **représentation**.

## Partie 4 · Le 0,507 — La meilleure version d'avant (0,507), reconstruite simplement

**Ce qu'était la V2 (record public 0,5072).**
- Un Transformer hybride : canaux relatifs + 4 embeddings de catégories, 3 blocs d'attention, pooling par attention.
- Une branche de statistiques, fusionnée avec la séquence.
- Un ensemble de deux variantes × deux graines.

⚠️ **C'est une reconstruction simplifiée**, pas le run exact :
- pas de biais « même ordre » ;
- pas de loss contrastive ;
- statistiques approchées ;
- budget court.

Elle sert de **point de comparaison honnête** pour V5, avec le même protocole et les mêmes données.

Durée : 2 graines × environ 8 min sur GPU.

In [ ]:
data = dict(lab.sequences('old'))
data['ctx'] = lab.table(core.OLD_STATS + ['levels'])[0]       # la V2 gardait aussi des niveaux dans sa branche statistique
EPOCHS = 3 if DEMO else 20
rows, hists, P = [], {}, {'valid': [], 'stress': []}
mods = []
for seed in [0, 1]:
    print('▶ graine', seed); t0 = time.time()
    m, h, norm, dev = T.train(models.V2Hybrid(data['cards'], data['cont'].shape[-1], data['ctx'].shape[1]), lab, data,
                              epochs=EPOCHS, lr=1e-3, seed=seed)
    out, p = T.evaluate(m, lab, data, norm, dev, f'Hybride V2 (graine {seed})', time.time() - t0)
    rows.append(out); hists[f'graine {seed}'] = h; mods.append((m, norm))
    for k in P: P[k].append(p[k])
ens = {'modèle': 'Ensemble des 2 graines (comme la V2)'}
for part in ['valid', 'stress']:
    p = np.mean(P[part], 0); y = lab.y[lab.split[part]]
    ens[part] = T.accuracy(y, p); ens[part + '_équilibré'] = T.accuracy(y, p, True)
rows.append(ens); res = pd.DataFrame(rows); res.round(4)

In [ ]:
RESULTS.append(res.assign(partie='Partie 4 · Le 0,507'))

In [ ]:
viz.curves(hists, 'V4 — hybride V2 reconstruit', 'v4_curves')
viz.compare(res, ('valid', 'stress'), 'V4 — la meilleure version d\'avant (reconstruction)', 'v4_compare')
viz.confusion(lab.y[lab.split['stress']], np.mean(P['stress'], 0), LABELS, 'V4 — confusion sur le stress', 'v4_confusion')

In [ ]:
# Soumission de contrôle (même règle que la V2 : argmax, sans équilibrage)
data_t = dict(lab.sequences('old', 'test')); data_t['ctx'] = lab.table(core.OLD_STATS + ['levels'], 'test')[0]
dev_t = T.OnDevice(data_t, lab.device)
Pt = np.mean([T.predict(m, dev_t, n, np.arange(len(lab.test_ids))) for m, n in mods], 0)
pd.DataFrame({'obs_id': lab.test_ids, 'eqt_code_cat': lab.classes[Pt.argmax(1)]}).to_csv('submission_V4_hybride_v2.csv', index=False)
print('submission_V4_hybride_v2.csv écrit — score public de référence de la vraie V2 : 0,5072')

**À retenir.**
- C'est le niveau de départ.
- Les scores de `valid` sont bien au-dessus du LB, parce que des fenêtres de jours voisins se ressemblent.
- `stress` est plus sévère.

V5 part d'un modèle **plus petit** que celui-ci et montre, étape par étape, ce qui fait gagner.

## Partie 5 · Améliorations — Un modèle simple, les améliorations une par une

On part d'un **petit CNN** (3 convolutions, environ 100 k paramètres) qui lit l'**ancienne représentation**. Puis on ajoute
**une seule chose à chaque étape**, en mesurant sur les mêmes fenêtres (`valid ∪ stress`) :

| Étape | Ce qu'on ajoute | Pourquoi (en une phrase) |
|---|---|---|
| 0 | Ancienne représentation | Grandeurs divisées par des médianes de la fenêtre : un spread de 1 tick et un de 5 ticks se ressemblent |
| 1 | **Tokens en ticks et en lots** | Le tick et le lot sont des conventions de marché propres à chaque titre, et elles ne changent pas d'une période à l'autre |
| 2 | **Augmentation de profondeur** | Le test est moins profond (prix plus élevés ?) : on apprend qu'un titre peut afficher plus ou moins d'actions |
| 3 | **Entraîner plus longtemps** | Les courbes montaient encore |
| 4 | **Moyenne de 3 graines** | Réduit le bruit d'un entraînement |
| 5 | **Équilibrage (Sinkhorn)** | Le décalage pousse le modèle vers les titres peu profonds ; on remet ~1/24 du test sur chaque titre |
| 6 | **Vote entre voisins** | Le test contient 20 fenêtres par titre et par jour : des fenêtres semblables votent ensemble |

Les étapes 5 et 6 utilisent le test sans labels (transductif). Durée totale : environ 15 à 20 min sur GPU.

In [ ]:
pool = np.r_[lab.split['valid'], lab.split['stress']]; y_pool = lab.y[pool]
acc = lambda P: float((P.argmax(1) == y_pool).mean())
E = 3 if DEMO else 20
ladder, hists, keep = [], {}, {}

def step(nom, kind, epochs, sigma=0., seed=0):
    data = lab.sequences(kind); t0 = time.time()
    m = models.TokenCNN(data['cards'], data['cont'].shape[-1], data['ctx'].shape[1])
    print(f'▶ {nom} — {models.n_params(m):,} paramètres')
    m, h, norm, dev = T.train(m, lab, data, epochs=epochs, depth_sigma=sigma, seed=seed)
    P = T.predict(m, dev, norm, pool); hists[nom] = h
    print(f'   accuracy valid ∪ stress = {acc(P):.4f} ({(time.time() - t0) / 60:.1f} min)')
    return m, norm, dev, P

In [ ]:
_, _, _, P0 = step('0 · ancienne représentation', 'old', E);                ladder.append(('0 · ancienne\nreprésentation', acc(P0)))
_, _, _, P1 = step('1 · + tokens ticks/lots', 'new', E);                     ladder.append(('1 · + tokens\nticks / lots', acc(P1)))
_, _, _, P2 = step('2 · + augmentation profondeur', 'new', E, .3);          ladder.append(('2 · + profondeur', acc(P2)))
m3, n3, d3, P3 = step('3 · + entraînement 2× plus long', 'new', 2 * E, .3); ladder.append(('3 · + 2× plus\nlong', acc(P3)))

In [ ]:
members = [(m3, n3, d3, P3)]
for seed in [1, 2]:
    members.append(step(f'4 · graine {seed}', 'new', 2 * E, .3, seed))
P4 = np.mean([x[3] for x in members], 0);       ladder.append(('4 · moyenne\n3 graines', acc(P4)))
P5 = sinkhorn_balance(P4);                      ladder.append(('5 · + équilibrage', acc(P5)))
# 6 · voisins : représentation = statistiques de régime ⊕ vecteurs internes du modèle, voisins cherchés dans le pool
Z = np.mean([T.predict(m, d, n, pool, features=True) for m, n, d, _ in members], 0)
W = TR.window_rep(lab.dir, lab.cfg, 'train', pool)
Zc = np.c_[W, TR.standardise(Z)]
P6 = sinkhorn_balance(TR.smooth(P4, TR.knn(Zc, 10), .5, 5)); ladder.append(('6 · + vote entre\nvoisins', acc(P6)))
pd.DataFrame(ladder, columns=['étape', 'accuracy valid ∪ stress']).round(4)

In [ ]:
viz.waterfall(ladder, 'V5 — chaque amélioration, une à la fois (même modèle simple, mêmes fenêtres)', 'v5_ladder', 'Accuracy valid ∪ stress')
viz.curves({k: v for k, v in hists.items() if not k.startswith('4')}, 'V5 — apprentissage des étapes 0 à 3', 'v5_curves')
viz.waterfall([('V2 (record\nprécédent)', .5072), ('SignatureNet\nblend 4', .5700), ('+ Sinkhorn', .5958), ('+ profondeur', .59999),
               ('+ 60 ép.,\nalignement', .6028), ('+ voisins', .6351)],
              'Scores PUBLICS réels du pipeline complet (pour comparaison)', 'v5_ladder_public', 'Accuracy publique')

In [ ]:
y_v = lab.y[lab.split['valid']]; nv = len(y_v)
viz.confusion(y_pool, P4, LABELS, 'Avant : moyenne de 3 graines (sans post-traitement)', 'v5_confusion_before')
viz.confusion(y_pool, P6, LABELS, 'Après : équilibrage + vote entre voisins', 'v5_confusion_after')
viz.recall_by_class(y_pool, {'0 · ancienne': P0, '4 · 3 graines': P4, '6 · + voisins': P6}, LABELS, 'Rappel par titre', 'v5_recall')
Zt_data = lab.sequences('new', 'test'); dt = T.OnDevice(Zt_data, lab.device)
Zt = T.predict(m3, dt, n3, np.arange(len(lab.test_ids)), features=True)
viz.embedding_map(T.predict(m3, d3, n3, lab.split['valid'], features=True), y_v, LABELS,
                  'Représentations du modèle : fenêtres valid (couleur = titre), test en gris', 'v5_tsne', extra=(Zt, 'test'))

In [ ]:
# Soumissions pour vérifier soi-même : étapes 4, 5 et 6 sur le test (voisins : k ×4 car le test a ~20 fenêtres par titre-jour)
idx_t = np.arange(len(lab.test_ids))
Pt = np.mean([T.predict(m, dt, n, idx_t) for m, n, _, _ in members], 0)
Zt_all = np.mean([T.predict(m, dt, n, idx_t, features=True) for m, n, _, _ in members], 0)
Zc_t = np.c_[TR.window_rep(lab.dir, lab.cfg, 'test', idx_t), TR.standardise(Zt_all)]
for nom, P in [('etape4_moyenne', Pt), ('etape5_equilibre', sinkhorn_balance(Pt)),
               ('etape6_voisins', sinkhorn_balance(TR.smooth(Pt, TR.knn(Zc_t, 40), .5, 5)))]:
    pd.DataFrame({'obs_id': lab.test_ids, 'eqt_code_cat': lab.classes[P.argmax(1)]}).to_csv(f'submission_V5_{nom}.csv', index=False)
    print('écrit : submission_V5_' + nom + '.csv')

**À retenir.**
1. Le gain le plus net vient de la **représentation** : les tokens en ticks et en lots.
2. Les deux post-traitements (**équilibrage**, **vote entre voisins**) ne demandent aucun réentraînement et pèsent lourd au LB : environ +2,6 et +3,2 points dans le pipeline complet.
3. Le modèle lui-même peut rester **petit**.

Sur `valid ∪ stress`, l'effet du vote entre voisins est **sous-estimé** : le pool contient environ 5 fenêtres par titre-jour, contre 20 dans le test.

## Synthèse — tous les modèles côte à côte
Même protocole, mêmes fenêtres. La cascade de la partie 5 et les scores publics réels figurent plus haut.

In [ ]:
allres = pd.concat(RESULTS, ignore_index=True)
cols = [c for c in ['partie', 'modèle', 'valid', 'valid_équilibré', 'stress', 'stress_équilibré', 'minutes'] if c in allres]
allres = allres[cols].sort_values('stress_équilibré', ascending=False)
allres.to_csv(viz.FIG_DIR / 'synthese_modeles.csv', index=False)
viz.compare(allres.assign(modèle=allres['partie'].str.split(' ').str[0] + ' · ' + allres['modèle']),
            ('valid_équilibré', 'stress_équilibré'), 'Tous les modèles (précision équilibrée)', 'synthese_modeles')
allres.round(4)

In [ ]:
import shutil
archive = shutil.make_archive(str(viz.FIG_DIR), 'zip', viz.FIG_DIR)
print('Figures :', sorted(p.name for p in viz.FIG_DIR.glob('*.png')), '→', archive)